In [ ]:
# Cell 1 - locate the FG-CLIP2 offline model dataset (the one used for Milvus ingestion)
from pathlib import Path

def find_model_root():
    known = Path('/kaggle/input/datasets/quanglongl040305/model2/aic_l28_offline_models/fgclip2')
    if known.is_dir():
        return known
    for cfg in Path('/kaggle/input').rglob('config.json'):
        p = cfg.parent
        if 'fgclip' in str(p).lower() and (any(p.glob('*.safetensors')) or any(p.glob('*.bin'))):
            return p
    raise RuntimeError('Attach the FG-CLIP2 model dataset (quanglongl040305/model2) and re-run.')

MODEL_ROOT = find_model_root()
print('model root:', MODEL_ROOT)


In [ ]:
# Cell 2 - load ZILLIZ_URI / ZILLIZ_TOKEN from a.env (private Kaggle Input)
from pathlib import Path

ENV_INPUT_FILENAME = 'a.env'

def load_a_env(path):
    settings = {}
    for raw in path.read_text(encoding='utf-8-sig').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        settings[key.strip().removeprefix('export ').strip()] = value.strip().strip('"').strip("'")
    missing = [k for k in ('ZILLIZ_URI', 'ZILLIZ_TOKEN') if not settings.get(k)]
    if missing:
        raise RuntimeError(f'{path} missing: {missing}')
    return settings

candidates = sorted({p.resolve() for p in Path('/kaggle/input').rglob(ENV_INPUT_FILENAME)})
if not candidates:
    raise RuntimeError(f'Attach the private Kaggle Input containing {ENV_INPUT_FILENAME}')
ENV = load_a_env(candidates[0])
print('a.env loaded from', candidates[0])


In [ ]:
# Cell 3 - load model + tokenizer (same settings as app/backend/encoders/fg_clip.py)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForCausalLM.from_pretrained(str(MODEL_ROOT), trust_remote_code=True, local_files_only=True).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_ROOT), trust_remote_code=True, local_files_only=True)
print('model loaded on', device)

MAX_TEXT_LENGTH = 64
TEXT_WALK_TYPE = 'short'

@torch.no_grad()
def encode_text(text):
    inputs = tokenizer([text], max_length=MAX_TEXT_LENGTH, padding='max_length',
                       truncation=True, return_tensors='pt').to(device)
    feats = model.get_text_features(**inputs, walk_type=TEXT_WALK_TYPE)
    feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
    return feats[0].float().cpu().numpy().tolist()


In [ ]:
# Cell 4 - AIC2026 P2 queries (English translations pre-computed)
QUERIES = [
 {
  "id": "query-p2-1-kis",
  "text": "A group of 5 people are playing next to a yellow animal. One of them hid an object that looked like a pumpkin. The man woke up and couldn't find the pumpkin, so he woke the animal up."
 },
 {
  "id": "query-p2-2-kis",
  "text": "The clip begins with a person using his phone to take a photo of a rhinoceros painting on the wall. The clip ends with a person taking a photo of three monkeys graffiti on a bridge"
 },
 {
  "id": "query-p2-3-kis",
  "text": "A yellow unicorn jumps or falls from above, close to a model of a small blue ship."
 },
 {
  "id": "query-p2-4-kis",
  "text": "Two young people are hanging a large blue banner, decorated with images of mountains, clouds and a road leading to school. On the banner there are also images of two children in difficult areas wearing yellow shirts."
 },
 {
  "id": "query-p2-5-kis",
  "text": "A man in a red shirt, wearing a white hat, is pouring water on his face. The frame has two cyclists, the one wearing a dark blue shirt is chasing the person wearing a black shirt mixed with orange."
 },
 {
  "id": "query-p2-6-kis",
  "text": "Video recorded from behind shows two young men driving a motorbike lying on the saddle and riding at high speed. The frame shows a blue car and another blue-shirted motorcyclist. There are 2 red circles surrounding the location of the 2 young men."
 },
 {
  "id": "query-p2-7-qa",
  "text": "Filmed from inside a self-driving car, the steering wheel is turned to turn the car to the right. Then the camera angle moves out, the white car turns left, in the upper corner of the frame there is a red sign with 6 Chinese characters. What is the number written on the white side of the car?"
 },
 {
  "id": "query-p2-8-trake--e1",
  "text": "The first scene features durian fruit in a Western orchard"
 },
 {
  "id": "query-p2-8-trake--e2",
  "text": "The first scene features mangosteen fruit in an orchard"
 },
 {
  "id": "query-p2-8-trake--e3",
  "text": "The first scene features grapefruit in an orchard"
 },
 {
  "id": "query-p2-8-trake--e4",
  "text": "The first scene has fruit growing in an orchard"
 },
 {
  "id": "query-p2-9-qa",
  "text": "In the cooking instruction video, the chef sequentially puts flavorings including green pepper, lime leaves and lemongrass inside the stomachs of a total of 4 fish. What kind of fish is this?"
 },
 {
  "id": "query-p2-10-kis",
  "text": "A chef prepares a dish in a pan, with white sausage pieces and green vegetables. The chef puts the chive flowers in the pan with the sausage and then uses a utensil to stir the ingredients. Long pieces of green chive flowers are mixed with white sausage pieces in the pan."
 },
 {
  "id": "query-p2-11-kis",
  "text": "The chef holds a long skewered ingredient and rolls it through the chopped green and red mixture. The ingredients are then transferred to a plate containing white flour to coat the outside. The chef holds the skewer and rotates the ingredients back and forth several times over the dough. Finally, the ingredients are covered with a layer of white powder and placed separately on a plate."
 },
 {
  "id": "query-p2-12-qa",
  "text": "The video describes the cake making process. The cake is purple in color, the ingredients inside are bean sprouts, carrots, and inside each cake is a lotus seed. How many cakes can this mold make at a time?"
 },
 {
  "id": "query-p2-13-kis",
  "text": "The chef stirs a mixture of ingredients in the pan. You can see chicken, red peppers, green peppers, peanuts and purple onions. She then turned off the heat and added lemon peel and lemon juice to the pan before pouring the mixture onto a plate."
 },
 {
  "id": "query-p2-14-kis",
  "text": "Close-up of a group of 3 cyclists moving close together, two riders wearing blue shirts and red and white helmets, next to a rider wearing a yellow shirt racing together. Below the red hat rider's hat strap is a white string hanging down near his neck."
 },
 {
  "id": "query-p2-15-kis",
  "text": "The scene introduces the ingredients of the dish in turn through 3 transitions: the camera tilts up and ends at the first seafood ingredient; Top-down close-up shot of the second seafood ingredient and then switching to colorful ingredients; Finally, there is a still shot of the entire scene of all the ingredients."
 },
 {
  "id": "query-p2-16-kis",
  "text": "In a country house with large windows, two women are crafting on a set of horse boards, behind which is a row of about 10 wooden cutting boards hung in a horizontal row."
 },
 {
  "id": "query-p2-17-kis",
  "text": "Stage with large 3D embossed lettering, covered in glitter with the content ANCIENT COLOR, placed at the front edge of the stage."
 },
 {
  "id": "query-p2-18-kis",
  "text": "The video was shot from behind the group leading the bicycle race, including 1 rider in front and 3 riders following, when the group turned right onto Ho Tung Mau Street at an intersection with a green light counting down to 13 seconds."
 },
 {
  "id": "query-p2-19-qa",
  "text": "The video recorded scenes of donors supporting an inn for the elderly, then switched to a scene of an old man chatting with a group of foreigners. What street is the inn mentioned in the video located on?"
 },
 {
  "id": "query-p2-20-kis",
  "text": "Geography lecture clip, has a data table about the urban network in Vietnam. This table shows the difference in urban distribution between regions by color: for the 3 regions with the most urban areas, the number representing the number of urban areas is printed in red, and for the 2 regions with the fewest urban areas, this number is printed in blue."
 },
 {
  "id": "query-p2-21-trake--e1",
  "text": "Two women work together to seal a carton"
 },
 {
  "id": "query-p2-21-trake--e2",
  "text": "Boxes of instant noodles and bread wraps are neatly arranged"
 },
 {
  "id": "query-p2-21-trake--e3",
  "text": "A man lifted a box of instant noodles and placed it on top of the pile of boxes"
 },
 {
  "id": "query-p2-21-trake--e4",
  "text": "Close-up of boxes of instant noodles stacked on trucks"
 },
 {
  "id": "query-p2-22-kis",
  "text": "In the cooking video, a white seafood ingredient is cut in perpendicular lines on both surfaces. The ingredients are then cut into sticks and put into a bowl, before being mixed with spices including wine, pepper and seasoning."
 },
 {
  "id": "query-p2-23-qa",
  "text": "The Biology question in the 2022 National High School Exam has a chart comparing the growth rates of plant species in coastal ecosystems. Ask plant species (II) to achieve the best growth rate when the habitat has a salinity of what part per thousand?"
 },
 {
  "id": "query-p2-24-kis",
  "text": "The finishing moment of a bicycle race. The Estonian athlete wearing a navy blue shirt led the group, released both hands from the handlebars, raised his arms high to celebrate victory while the bike was still moving forward. Right behind him are an athlete wearing a yellow shirt and an athlete wearing an orange shirt rushing towards the finish line."
 },
 {
  "id": "query-p2-25-kis",
  "text": "A scene of a male teacher wearing glasses and a short-sleeved striped shirt appears in the lower left corner and uses his hands to make illustrative gestures while lecturing. The frame contains an illustration of a young girl wearing glasses, wearing a white shirt, sitting cross-legged on a gray sofa, holding a glass of water and looking at an open laptop on her lap."
 },
 {
  "id": "query-p2-26-kis",
  "text": "The lecture slide includes: a group of white 3D human characters surrounding a red character in the middle. Two male cartoon characters are in a tug-of-war competition position, confronting each other with a rope."
 },
 {
  "id": "query-p2-27-qa",
  "text": "A unicorn is performing, the columns for the unicorn to perform are labeled with numbers. Behind it is a spiral-shaped dragon model. During the first 16 seconds of the video, which number from 1 to 8 is not visible from the camera's perspective?"
 },
 {
  "id": "query-p2-28-qa",
  "text": "A bowl of porridge has been completely cooked and decorated. Next to the bowl of porridge, there is a small black bowl containing an orange topping, the texture is a bit like small fibers. This topping was previously sprinkled on a bowl of porridge. Ask what animal's meat is the topping in the video?"
 },
 {
  "id": "query-p2-29-qa",
  "text": "The scene lists the ingredients to cook a dish. The background image includes a plate of meat, a bunch of fresh green leaves, a packet of seasoning powder, a small glass jar containing coconut milk, curry powder, dried mushrooms, lemongrass and red chili. The ingredient list that appears includes 9 ingredients. How much does the meat weigh in the ingredients list?"
 },
 {
  "id": "query-p2-30-qa",
  "text": "A girl wearing a white apron, next to a vase of purple galangal flowers. Then this girl placed 4 X's on a white plate. X's are the ingredients for the dish in this episode. Then this person held up 2 X's. What is X?"
 }
]

print(len(QUERIES), 'queries ready')


In [ ]:
# Cell 5 - search Milvus (visual_embedding only; caption_embedding is all-zero in this collection)
from pymilvus import MilvusClient, AnnSearchRequest
import json, time

client = MilvusClient(uri=ENV['ZILLIZ_URI'], token=ENV['ZILLIZ_TOKEN'])
COLLECTION = 'BoldSearch'
TOP_K = 100

results = {}
for q in QUERIES:
    t0 = time.time()
    vec = encode_text(q['text'])
    req = AnnSearchRequest(data=[vec], anns_field='visual_embedding',
                           param={'metric_type': 'COSINE', 'params': {}}, limit=TOP_K)
    hits = client.hybrid_search(collection_name=COLLECTION, reqs=[req],
                                ranker=None, limit=TOP_K,
                                output_fields=['video_id', 'frame_id', 'shot_id'])
    rows = []
    dists = []
    for group in (hits if isinstance(hits, list) else [hits]):
        for h in (group if isinstance(group, list) else [group]):
            ent = h.get('entity') or {}
            rows.append({'video_id': ent.get('video_id'), 'frame_id': ent.get('frame_id'),
                         'shot_id': ent.get('shot_id'), 'distance': float(h.get('distance', 0))})
            dists.append(float(h.get('distance', 0)))
    results[q['id']] = rows
    if dists:
        spread = max(dists) - min(dists)
        health = 'OK (discriminative)' if spread > 0.05 else 'FLAT - model mismatch suspected!'
        print(f"{q['id']}: top1={rows[0]['video_id']}#{rows[0]['frame_id']} d={dists[0]:.3f} spread={spread:.3f} {health} [{time.time()-t0:.1f}s]")

with open('/kaggle/working/p2_results.json', 'w', encoding='utf-8') as fh:
    json.dump(results, fh, ensure_ascii=False)
print('saved /kaggle/working/p2_results.json')
